# RAG Pipeline from Scratch

This notebook demonstrates how to build a simple Retrieval-Augmented Generation (RAG) pipeline from scratch, without relying on frameworks like LangChain or LlamaIndex. It uses the `google-genai` library for embeddings and a basic `FaissRetriever` for vector search.

## 1. Setup and Package Installation

In [1]:
!pip install -q -U "google-genai>=1.0.0"
!pip install python_dotenv
!pip install pymupdf
!pip install pdfplumber
!pip install faiss-cpu
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.8/241.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 74.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 55.7 MB/s eta 0:00:00


## 2. API Key and Model Initialization

In [21]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv("config.env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.5-flash-lite"
EVALUATION_MODEL = "gemini-2.5-pro"

## 3. Data Loading and Processing

We'll use the `UnstructuredDataLoader` from the provided `util.py` to extract text from a PDF and chunk it for processing.

In [3]:
from util import UnstructuredDataLoader

# Assuming 'Table_Reconstruction.pdf' is available in the same directory
unstr_loader = UnstructuredDataLoader()
text = unstr_loader.extract_text_from_pdf("/content/ETF_vs_MutualFunds_expanded.pdf")

# Set a chunk size and overlap for text chunking
unstr_loader.max_chunk_size = 1000
chunks = unstr_loader.chunk_text(text[0], overlap=100)

print(f"Created {len(chunks)} text chunks.")

Created 5 text chunks.


In [4]:
tables_md, tables_df = unstr_loader.extract_tables_from_pdf("/content/ETF_vs_MutualFunds_expanded.pdf")

In [5]:
chunks.extend(tables_md)

In [6]:
len(chunks)

7

## 4. Embedding and Vector Store

In [7]:
import numpy as np
from util import FaissRetriever

client = genai.Client(api_key=GEMINI_API_KEY)

# Generate embeddings for all text chunks
result = client.models.embed_content(model=EMBEDDING_MODEL, contents=chunks)
dim = len(result.embeddings[0].values)
embeddings = np.empty((len(chunks), dim))
for i, embedding in enumerate(result.embeddings):
  embeddings[i] = np.array(embedding.values)


In [8]:

# Initialize and add embeddings to the Faiss retriever
retriever = FaissRetriever(embedding_dim=dim)
retriever.add(embeddings, chunks)

print(f"Faiss index created with {retriever.index.ntotal} vectors.")

Faiss index created with 7 vectors.


## 5. Retrieval-Augmented Generation (RAG)

Now, let's define a function to perform RAG. It will:
1. Take a user query.
2. Embed the query.
3. Use the retriever to find relevant text chunks.
4. Combine the query and the retrieved chunks into a single prompt.
5. Use a large language model to generate a final answer based on the combined information.

In [9]:
def rag_pipeline(query):
    """
    Performs a simple RAG pipeline to answer a query.
    """
    # Step 1 & 2: Embed the query and retrieve relevant chunks
    query_embed = client.models.embed_content(model=EMBEDDING_MODEL, contents=query)
    query_embedding = np.array(query_embed.embeddings[0].values).reshape(-1, dim)
    retrieved_chunks = retriever.retrieve(query_embedding)

    # Step 3 & 4: Create the prompt with retrieved context
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""
    You are a helpful assistant. Use the following pieces of context to answer the question at the end.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.

    Context:
    {context}

    Question: {query}

    Answer:
    """

    # Step 5: Generate the answer using the language model
    response = client.models.generate_content(
      model=GENERATION_MODEL,
      contents=prompt,
    )
    return response.text


In [10]:

query = "What is the advantage of ETF over mutual funds?"
answer = rag_pipeline(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is the advantage of ETF over mutual funds?
Answer: ETFs offer several advantages over mutual funds, including:

*   **Liquidity:** ETFs trade on exchanges throughout the day, allowing investors to buy and sell at market prices during market hours. This contrasts with most mutual funds, which are priced only once per day after the market closes.
*   **Costs and Fees:** ETFs generally have lower expense ratios compared to actively managed mutual funds and are often lower or competitive with index mutual funds. They also typically avoid 12b-1 distribution fees.
*   **Tax Efficiency:** ETFs have a tax-efficient structure due to their in-kind creation/redemption mechanism. This process reduces the need for fund managers to sell underlying securities, which lowers realized capital gains distributions to shareholders.
*   **Transparency:** While not explicitly detailed in the provided text, ETFs are generally known for their transparency in holdings.


In [11]:

query = "What are different kind of ETFs?"
answer = rag_pipeline(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What are different kind of ETFs?
Answer: The different kinds of ETFs include:

*   Broad-market index ETFs
*   Sector ETFs
*   Bond ETFs
*   Commodity ETFs
*   Thematic ETFs
*   Actively managed ETFs


In [12]:

query = "Are there any risks of investing in ETF?"
answer = rag_pipeline(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: Are there any risks of investing in ETF?
Answer: Yes, there are risks associated with investing in ETFs. These include tracking error, liquidity of the underlying assets (especially for niche or leveraged ETFs), counterparty risks for synthetic ETFs, fee structure, and the potential for bid/ask spreads to increase trading costs.


In [13]:
tables_df[0]

,Fund Type,Index (avg),Active (avg)
0,ETF,0.48%,0.69%
1,Mutual Fund,0.60%,0.89%


In [14]:
tables_df[1]

,Study / Source,Finding
0,Bank of America (reported in FT),Estimated US investors saved ~$250 bi
1,Industry reports,Lower tax drag and structural advantag


In [15]:

query = "What was the finding according to industry reports"
answer = rag_pipeline(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What was the finding according to industry reports
Answer: According to industry reports, lower tax drag and structural advantages reduce taxable events.


In [16]:

query = "What is the active average of ETF"
answer = rag_pipeline(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is the active average of ETF
Answer: 0.69%


### Evaluation of the results

`keyword_evaluation` function as requested, implementing the logic to check for the presence of all keywords in the generated answer string.



In [17]:
def keyword_evaluation(answer, keywords):
    """
    Evaluates the RAG pipeline's response based on keyword matching.

    Args:
        answer: The generated answer string from the RAG pipeline.
        keywords: A list of keywords to check for in the answer.

    Returns:
        True if all keywords are found in the answer (case-insensitive),
        False otherwise.
    """
    answer_lower = answer.lower()
    for keyword in keywords:
        if keyword.lower() not in answer_lower:
            return False
    return True

`similarity_evaluation` function to calculate the cosine similarity between the generated answer and each retrieved chunk using embeddings.



In [18]:
from sklearn.metrics.pairwise import cosine_similarity

def similarity_evaluation(answer, retrieved_chunks, client):
    """
    Evaluates the RAG pipeline's response by calculating the semantic similarity
    between the generated answer and the retrieved sources using embeddings.

    Args:
        answer: The generated answer string from the RAG pipeline.
        retrieved_chunks: A list of strings representing the retrieved context chunks.
        client: The google.genai client object for embedding.

    Returns:
        A list of similarity scores, where each score represents the cosine similarity
        between the answer embedding and a corresponding retrieved chunk embedding.
    """
    # Embed the answer
    answer_embed = client.models.embed_content(model=EMBEDDING_MODEL, contents=answer)
    answer_embedding = np.array(answer_embed.embeddings[0].values).reshape(1, -1)

    # Embed the retrieved chunks
    chunks_embed = client.models.embed_content(model=EMBEDDING_MODEL, contents=retrieved_chunks)
    chunk_embeddings = np.array([emb.values for emb in chunks_embed.embeddings])

    # Calculate cosine similarity between the answer and each chunk
    similarity_scores = cosine_similarity(answer_embedding, chunk_embeddings)[0]

    return similarity_scores.tolist()


`llm_judge_evaluation` function to create a prompt for the LLM to evaluate the RAG answer based on the query and context, and use the provided client to get the LLM's evaluation.



In [23]:
def llm_judge_evaluation(query, context, answer, client):
    """
    Uses an LLM to assess the quality and relevance of the RAG pipeline's answer
    based on the original query and the retrieved context.

    Args:
        query: The original user query string.
        context: The retrieved context as a single string or list of chunks.
        answer: The generated answer string from the RAG pipeline.
        client: The google.genai client object.

    Returns:
        The LLM's response containing the evaluation.
    """
    if isinstance(context, list):
        context = "\n\n".join(context)

    prompt = f"""
    You are an expert evaluator of question answering systems.
    Your task is to assess the quality of an answer generated by a RAG pipeline
    based on a given query and the context retrieved by the system.

    Evaluate the answer based on the following criteria:
    1. Relevance: Is the answer directly relevant to the query?
    2. Consistency: Is the answer consistent with the information provided in the context?
    3. Completeness: Does the answer address all parts of the query based on the context?
    4. Conciseness: Is the answer free of unnecessary information?
    5. Accuracy: Is the information in the answer factually correct based *only* on the context provided?

    Provide output in a scale of 1 to 5,
    where 1 is poor and 5 is excellent, avoid providing any explanation until unless asked explicitly

    Query: {query}

    Context:
    {context}

    Generated Answer:
    {answer}

    Evaluation:
    """

    response = client.models.generate_content(
      model=EVALUATION_MODEL,
      contents=prompt,
    )
    return response.text

In [24]:
# Modify the rag_pipeline function to return retrieved chunks
def rag_pipeline(query):
    """
    Performs a simple RAG pipeline to answer a query and returns the answer and retrieved chunks.
    """
    # Step 1 & 2: Embed the query and retrieve relevant chunks
    query_embed = client.models.embed_content(model=EMBEDDING_MODEL, contents=query)
    query_embedding = np.array(query_embed.embeddings[0].values).reshape(-1, dim)
    retrieved_chunks = retriever.retrieve(query_embedding)

    # Step 3 & 4: Create the prompt with retrieved context
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""
    You are a helpful assistant. Use the following pieces of context to answer the question at the end.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.

    Context:
    {context}

    Question: {query}

    Answer:
    """

    # Step 5: Generate the answer using the language model
    response = client.models.generate_content(
      model=GENERATION_MODEL,
      contents=prompt,
    )
    return response.text, retrieved_chunks

# 1. Define a query
query_to_evaluate = "What are the advantages of ETFs over mutual funds and what are the risks?"

# 2. Call the rag_pipeline function
generated_answer, retrieved_chunks_for_eval = rag_pipeline(query_to_evaluate)

print(f"Query: {query_to_evaluate}")
print(f"Generated Answer: {generated_answer}")
print(f"Retrieved Chunks (first 2): {retrieved_chunks_for_eval[:2]}...")

# 3. Define expected keywords
expected_keywords = ["liquidity", "cost", "tax efficiency", "risks", "tracking error", "liquidity of underlying assets"]

# 4. Call keyword_evaluation and print the result
keyword_match_result = keyword_evaluation(generated_answer, expected_keywords)
print(f"\nKeyword Evaluation Result: {keyword_match_result}")

# 5. Call similarity_evaluation and print the result
similarity_scores = similarity_evaluation(generated_answer, retrieved_chunks_for_eval, client)
print(f"Similarity Evaluation Scores: {similarity_scores}")

# 6. Call llm_judge_evaluation and print the result
llm_evaluation_result = llm_judge_evaluation(query_to_evaluate, retrieved_chunks_for_eval, generated_answer, client)
print(f"\nLLM Judge Evaluation:\n{llm_evaluation_result}")

Query: What are the advantages of ETFs over mutual funds and what are the risks?
Generated Answer: Here are the advantages of ETFs over mutual funds, along with their associated risks:

**Advantages of ETFs over Mutual Funds:**

*   **Cost Efficiency:** ETFs generally have lower expense ratios compared to actively managed mutual funds and often competitive or lower fees than index mutual funds. They also typically avoid 12b-1 distribution fees that some mutual funds charge.
*   **Tax Efficiency:** In taxable accounts, ETFs are more tax-efficient due to their in-kind creation/redemption process. This process reduces the need for fund managers to sell underlying securities, which lowers realized capital gains distributions to shareholders.
*   **Intraday Trading and Liquidity:** ETFs trade on stock exchanges throughout the trading day, allowing investors to buy and sell at market prices during market hours. This contrasts with most mutual funds, which are priced only once per day after m

In [25]:
# Evaluate a question from the tables
query_from_table = "What is the active average of Mutual Fund according to the table?"

# Call the rag_pipeline function
generated_answer_table, retrieved_chunks_table = rag_pipeline(query_from_table)

print(f"Query: {query_from_table}")
print(f"Generated Answer: {generated_answer_table}")
print(f"Retrieved Chunks (first 2): {retrieved_chunks_table[:2]}...")

# Define expected keywords for the table question
expected_keywords_table = ["0.89%", "Mutual Fund", "Active", "average"]

# Call keyword_evaluation and print the result
keyword_match_result_table = keyword_evaluation(generated_answer_table, expected_keywords_table)
print(f"\nKeyword Evaluation Result (Table Question): {keyword_match_result_table}")

# Call similarity_evaluation and print the result
similarity_scores_table = similarity_evaluation(generated_answer_table, retrieved_chunks_table, client)
print(f"Similarity Evaluation Scores (Table Question): {similarity_scores_table}")

# Call llm_judge_evaluation and print the result
llm_evaluation_result_table = llm_judge_evaluation(query_from_table, retrieved_chunks_table, generated_answer_table, client)
print(f"\nLLM Judge Evaluation (Table Question):\n{llm_evaluation_result_table}")

Query: What is the active average of Mutual Fund according to the table?
Generated Answer: 0.89%
Retrieved Chunks (first 2): ['--- Table 1 on Page 3 ---\n| **Fund Type** | **Index (avg)** | **Active (avg)** |\n| --- | --- | --- |\n| ETF | 0.48% | 0.69% |\n| Mutual Fund | 0.60% | 0.89% |', '12b-1 distribution fees that some\nmutual funds charge.\nTable 1: Typical average expense ratios (percentage)\nFund Type\nIndex (avg)\nActive (avg)\nETF\n0.48%\n0.69%\nMutual Fund\n0.60%\n0.89%\n\n\n--- Page 4 ---\nTax Efficiency and Investor Savings\nOne of the major advantages of ETFs in taxable accounts is their tax-efficient structure. The in-kind\ncreation/redemption process reduces the need for the fund manager to sell underlying securities —\nlowering realized capital gains distributions to shareholders. Industry studies estimate meaningful tax\nsavings for ETF investors compared with mutual fund investors.\nStudy / Source\nFinding\nBank of America (reported in FT)\nEstimated US investors save